In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob


In [ ]:
def cargar_y_preparar_datos(filepath_or_pattern: str) -> pd.DataFrame:
    """Carga uno o múltiples archivos parquet/csv ya normalizados y genera variables temporales para EDA."""
    archivos = sorted(glob.glob(filepath_or_pattern))

    if not archivos:
        raise FileNotFoundError(
            f'No se encontraron archivos con el patrón: {filepath_or_pattern}'
        )

    lista_dfs = [
        pd.read_parquet(f) if f.endswith('.parquet') else pd.read_csv(f)
        for f in archivos
    ]
    df = pd.concat(lista_dfs, ignore_index=True)

    # 1. Asegurar tipo datetime en columna normalizada 'inicio_del_viaje'
    if not pd.api.types.is_datetime64_any_dtype(df['inicio_del_viaje']):
        df['inicio_del_viaje'] = pd.to_datetime(
            df['inicio_del_viaje'], errors='coerce'
        )

    # 2. Generación de variables derivadas para el análisis exploratorio
    df['fecha_hora'] = df['inicio_del_viaje'].dt.floor('h')
    df['hora'] = df['fecha_hora'].dt.hour
    df['dia_semana'] = df['fecha_hora'].dt.day_name()

    return df


def graficar_demanda_por_hora(df: pd.DataFrame) -> None:
    """Genera el gráfico de barras de viajes totales por hora del día."""
    fig, ax = plt.subplots(figsize=(10, 4))
    conteo_hora = df.groupby('hora').size()

    sns.barplot(
        x=conteo_hora.index, y=conteo_hora.values, color='skyblue', ax=ax
    )
    ax.set_title('Demanda Total por Hora del Día (P1)')
    ax.set_xlabel('Hora del día')
    ax.set_ylabel('Cantidad de viajes')
    plt.tight_layout()
    plt.show()


def graficar_top_estaciones(df: pd.DataFrame, top_n: int = 10) -> None:
    """Muestra las estaciones con mayor volumen de origen de viajes (usando 'origen_id')."""
    top = df['origen_id'].value_counts().head(top_n)

    fig, ax = plt.subplots(figsize=(10, 4))
    
    sns.barplot(
        x=top.index.astype(str), 
        y=top.values, 
        hue=top.index.astype(str),
        palette='Blues_r', 
        legend=False,
        ax=ax
    )
    
    ax.set_title(f'Top {top_n} Estaciones con Mayor Demanda')
    ax.set_xlabel('ID de Estación (origen_id)')
    ax.set_ylabel('Viajes Totales')
    plt.tight_layout()
    plt.show()


def calcular_resumen_demanda_horaria(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula estadísticas descriptivas de la demanda agrupada por estación y hora."""
    demanda = (
        df.groupby(['origen_id', 'fecha_hora'])
        .size()
        .reset_index(name='demanda')
    )
    resumen = demanda['demanda'].describe().to_frame(name='Estadística')
    return resumen

def graficar_demanda_semana_vs_finde(df: pd.DataFrame) -> None:
    """Compara la curva horaria entre días laborales y fines de semana."""
    df_temp = df.copy()
    df_temp['es_fin_semana'] = df_temp['inicio_del_viaje'].dt.dayofweek.isin([5, 6]).map(
        {True: 'Fin de Semana', False: 'Día Laboral'}
    )
    
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.lineplot(
        data=df_temp.groupby(['hora', 'es_fin_semana']).size().reset_index(name='viajes'),
        x='hora',
        y='viajes',
        hue='es_fin_semana',
        marker='o',
        ax=ax
    )
    ax.set_title('Perfil de Demanda Horaria: Día Laboral vs Fin de Semana')
    ax.set_xlabel('Hora del Día')
    ax.set_ylabel('Total de Viajes')
    plt.tight_layout()
    plt.show()


def graficar_heatmap_hora_dia(df: pd.DataFrame) -> None:
    """Genera un mapa de calor de Horas vs Días de la semana."""
    dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    
    pivot = df.groupby(['hora', 'dia_semana']).size().unstack(fill_value=0)
    pivot = pivot.reindex(columns=[d for d in dias if d in pivot.columns])
    
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(pivot, cmap='YlGnBu', ax=ax, cbar_kws={'label': 'Cantidad de Viajes'})
    ax.set_title('Matriz de Calor de Demanda (Hora vs Día de la Semana)')
    ax.set_xlabel('Día de la Semana')
    ax.set_ylabel('Hora del Día')
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 2: Cargar datos
df = cargar_y_preparar_datos('../data/processed/*.parquet')
print(f'Total de registros cargados: {len(df):,}')
print(f'Rango de fechas: {df["fecha_hora"].min()} a {df["fecha_hora"].max()}')

In [ ]:
# Cell 3: Visualizaciones modulares
graficar_demanda_por_hora(df)
graficar_top_estaciones(df, top_n=15)

In [ ]:
# Cell 4: Resumen estadístico
calcular_resumen_demanda_horaria(df)

In [ ]:
graficar_demanda_semana_vs_finde(df)


In [ ]:
graficar_heatmap_hora_dia(df)

## Insights encontrados:

1. **Bimodalidad en días laborales:** Los picos de las 8:00 am y 6:00 pm de lunes a viernes confirman que el principal uso es traslados casa-trabajo/escuela.
2. **Unimodalidad uso recreativo en fin de semana:** En sabado y domingo desaparecen los picos de hora punta y la demanda es mas distribuida y constante durante desde 11:00 am a 3:00 pm
3. **Justificación de features:** La variable es_fin_de_semana e historicos por hora (lags) serán determinantes para que el modelo no sobrestime viajes en un sábado a las 8:00 AM.
 